In [1]:
!pip install langchain langchain-text-splitters


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import re
import json
import os

INPUT_FILE = 'DATA_WEB_FIT.md'
OUTPUT_JSON = 'CHUNKED_RECURSIVE_DATA.json'

def clean_markdown_content(text):
    # Remove author tags
    text = re.sub(r'Tác giả\s*:\s*', '', text)

    # Remove image markdown: ![](url) hoặc ![alt](url)
    text = re.sub(r'!\[[^\]]*\]\([^)]+\)', ' ', text)

    # Remove topic/category links but keep article text links
    text = re.sub(r'\[[^\]]+\]\(/\?TopicId=[^)]+\)', ' ', text)

    # Convert markdown links to plain text:
    # [Giới thiệu Khoa](url) -> Giới thiệu Khoa
    text = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', text)

    # Remove common website noise
    trash_keywords = [
        "Góp ý",
        "Họ và tên:",
        "RadEditor - HTML",
        "CÁC ĐỐI TÁC - KHÁCH HÀNG"
    ]

    for kw in trash_keywords:
        if kw in text:
            text = text.split(kw)[0]

    # Remove leftover image/file paths
    text = re.sub(r'/Resources/[^\s\)]*', ' ', text)
    text = re.sub(r'/Services/[^\s\)]*', ' ', text)

    # Remove lonely brackets and markdown residues
    text = re.sub(r'[\[\]\(\)]', ' ', text)

    # Normalize spaces and newlines
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n\s*\n+', '\n\n', text)

    return text.strip()

In [3]:
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False
)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=80,
    separators=[
        "\n\n",
        "\n",
        ". ",
        "! ",
        "? ",
        "; ",
        ", ",
        " ",
        ""
    ],
    keep_separator=True
)

In [4]:
def is_valid_chunk(text):
    text = text.strip()

    if len(text) < 120:
        return False

    noise_patterns = [
        "CÁC ĐỐI TÁC - KHÁCH HÀNG",
        "Album Hình Ảnh Hoạt Động",
        "Tất cả",
        "tinkhac",
        "Resources/Images",
        "Services/GetArticleImage",
        "Default.aspx?ModuleId",
    ]

    if any(pattern in text for pattern in noise_patterns):
        return False

    # Loại chunk gần như chỉ là danh sách link/ngày tháng
    article_id_count = text.count("ArticleId")
    if article_id_count >= 5:
        return False

    # Loại chunk có quá nhiều dòng ngắn dạng menu/list
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if len(lines) >= 8:
        short_lines = [line for line in lines if len(line) < 60]
        if len(short_lines) / len(lines) > 0.75:
            return False

    return True

In [5]:
print(f"Processing '{INPUT_FILE}'...")

try:
    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        raw_data = f.read()

    parts = re.split(r'={60}\nURL: (.*?)\n={60}\n', raw_data)

    final_chunks = []
    skipped_chunks = 0
    skipped_pages = 0

    for i in range(1, len(parts), 2):
        url = parts[i].strip()
        raw_content = parts[i + 1].strip()

        # Bỏ qua trang chủ vì thường gom nhiều chuyên mục lẫn nhau
        if url.rstrip("/") == "http://fit.hcmute.edu.vn":
            skipped_pages += 1
            continue

        clean_content = clean_markdown_content(raw_content)

        if not clean_content or len(clean_content) < 200:
            skipped_pages += 1
            continue

        md_header_splits = markdown_splitter.split_text(clean_content)
        recursive_splits = text_splitter.split_documents(md_header_splits)

        for chunk in recursive_splits:
            chunk_text = chunk.page_content.strip()

            if not is_valid_chunk(chunk_text):
                skipped_chunks += 1
                continue

            meta = dict(chunk.metadata)
            meta["source_url"] = url

            final_chunks.append({
                "page_content": chunk_text,
                "metadata": meta
            })

    with open(OUTPUT_JSON, 'w', encoding='utf-8') as f_out:
        json.dump(final_chunks, f_out, ensure_ascii=False, indent=4)

    print(f"Success: Processed {len(parts)//2} pages into {len(final_chunks)} clean chunks.")
    print(f"Skipped pages: {skipped_pages}")
    print(f"Skipped chunks: {skipped_chunks}")
    print(f"Output saved to: {OUTPUT_JSON}")

except FileNotFoundError:
    print(f"Error: File '{INPUT_FILE}' not found.")
except Exception as e:
    print(f"An error occurred: {e}")

Processing 'DATA_WEB_FIT.md'...
Success: Processed 191 pages into 561 clean chunks.
Skipped pages: 68
Skipped chunks: 118
Output saved to: CHUNKED_RECURSIVE_DATA.json


In [6]:
for i, chunk in enumerate(final_chunks[:3]):
    print("=" * 80)
    print("Chunk", i + 1)
    print("URL:", chunk["metadata"].get("source_url"))
    print(chunk["page_content"][:1000])

Chunk 1
URL: http://fit.hcmute.edu.vn/Default.aspx?ArticleId=bafd29bd-264e-4a5d-a92f-afe11c777e95
Khoa Công nghệ thông tin được thành lập năm 2001 dựa trên Trung tâm Tin học thành lập năm 1990 . Khoa chịu trách nhiệm đào tạo kỹ sư và giáo viên và các hoạt động nghiên cứu về Công nghệ thông tin. Khoa hiện có 4 bộ môn là Công nghệ phần mềm, Hệ thống Thông tin, Mạng và An ninh mạng và Trí tuệ Nhân tạo
Chunk 2
URL: http://fit.hcmute.edu.vn/Default.aspx?ArticleId=bafd29bd-264e-4a5d-a92f-afe11c777e95
. Khoa có 3 chương trình đào tạo đại học bao gồm Công nghệ thông tin, An toàn Thông tin và Kỹ thuật dữ liệu và 1 chương trình Thạc sĩ Khoa học Máy tính. Khoa có 1 phòng máy chủ, 6 phòng máy tính và 6 phòng lab cho Kỹ thuật phần mềm, Hệ thống thông tin, An ninh mạng, MacOS, Trí tuệ Nhân tạo và Kỹ thuật dữ liệu với tổng cộng 460 máy tính và 12 máy chủ.
Chunk 3
URL: http://fit.hcmute.edu.vn/Default.aspx?ArticleId=bafd29bd-264e-4a5d-a92f-afe11c777e95
Khoa đã nhận được bằng khen của Bộ trưởng Bộ Giáo